# Ejercicios de Limpieza y Preprocesamiento de Datos

<div style="background: linear-gradient(135deg, #2c3e50 0%, #3498db 100%); padding: 30px; border-radius: 10px; color: white; margin-bottom: 20px;">
<h2 style="color: white; margin-top: 0;">30 ejercicios practicos para dominar la limpieza de datos</h2>
<p style="font-size: 16px;">Cada ejercicio incluye enunciado, resolucion detallada y codigo con visualizacion antes/despues.</p>
</div>

---

## Indice de Ejercicios

| # | Bloque | Ejercicio |
|---|--------|-----------|
| **Bloque 1** | **Exploracion** | |
| 1 | Exploracion | Perfilar df_dirty: shape, dtypes, nulls, describe |
| 2 | Exploracion | Identificar columnas con >5% nulls, tipos incorrectos, constantes |
| 3 | Exploracion | Generar reporte de calidad (tabla con % nulls, tipo, n_unique por columna) |
| 4 | Exploracion | Heatmap de missing values |
| **Bloque 2** | **Valores Nulos** | |
| 5 | Valores Nulos | Imputar 'edad' con media vs mediana -- comparar histogramas |
| 6 | Valores Nulos | Imputar 'departamento' (categorica) con moda |
| 7 | Valores Nulos | Serie temporal simulada, imputar con forward-fill |
| 8 | Valores Nulos | Imputar 'salario' por grupo (media por departamento) |
| 9 | Valores Nulos | KNN Imputer en columnas numericas -- comparar con media |
| 10 | Valores Nulos | Crear columna 'salario_missing' como flag antes de imputar |
| **Bloque 3** | **Duplicados** | |
| 11 | Duplicados | Detectar y eliminar duplicados exactos |
| 12 | Duplicados | Detectar duplicados aproximados en 'nombre' |
| 13 | Duplicados | Duplicados por subconjunto ['nombre', 'departamento'] |
| **Bloque 4** | **Outliers** | |
| 14 | Outliers | Detectar outliers en 'salario' con IQR, boxplot |
| 15 | Outliers | Detectar outliers en 'edad' con Z-score |
| 16 | Outliers | Winsorizar 'salario' al percentil 1-99 vs eliminar |
| 17 | Outliers | Isolation Forest en ['edad', 'salario', 'evaluacion'] |
| **Bloque 5** | **Tipos y Validacion** | |
| 18 | Tipos y Validacion | Limpiar columna "$1,234.56" y convertir a float |
| 19 | Tipos y Validacion | Parsear 'fecha_ingreso' con formatos mixtos |
| 20 | Tipos y Validacion | Validar rangos: edad [18,70], salario > 0, evaluacion [1,10] |
| **Bloque 6** | **Limpieza de Strings** | |
| 21 | Limpieza de Strings | Normalizar 'nombre': strip, title case, espacios multiples |
| 22 | Limpieza de Strings | Extraer dominio de emails con regex |
| 23 | Limpieza de Strings | Estandarizar 'departamento': title case, corregir inconsistencias |
| **Bloque 7** | **Encoding** | |
| 24 | Encoding | One-Hot Encoding de 'ciudad' con drop_first |
| 25 | Encoding | Ordinal Encoding de 'satisfaccion' |
| 26 | Encoding | Target Encoding de 'departamento' con media de evaluacion |
| **Bloque 8** | **Escalado** | |
| 27 | Escalado | Comparar MinMax vs Standard vs Robust en 'salario' |
| 28 | Escalado | ColumnTransformer: escalar numericas + one-hot categoricas |
| **Bloque 9** | **Pipelines** | |
| 29 | Pipelines | Pipeline completo: imputar + escalar numericas, imputar + one-hot categoricas |
| 30 | Pipelines | Limpieza end-to-end: df_dirty a df_clean listo para modelo |

In [ ]:
# === IMPORTS Y CONFIGURACION ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import (StandardScaler, MinMaxScaler, RobustScaler,
                                   MaxAbsScaler, LabelEncoder, OneHotEncoder,
                                   OrdinalEncoder, PowerTransformer)
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from scipy import stats
from IPython.display import display, HTML

# Estilos
display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>.output_result { max-width:100% !important; }</style>"))
plt.style.use('seaborn-v0_8-whitegrid')

C_PRIMARY = '#3498db'
C_DANGER = '#e74c3c'
C_SUCCESS = '#2ecc71'
C_DARK = '#2c3e50'
C_ORANGE = '#f39c12'

# === CREAR DATASET SINTETICO "SUCIO" ===
np.random.seed(42)
n = 500

df_dirty = pd.DataFrame({
    'id': range(1, n + 1),
    'nombre': np.random.choice(
        ['Juan Garcia', 'MARIA LOPEZ', '  pedro martinez  ', 'Ana Torres',
         'LUIS sanchez', None, 'pedro Martinez'], n),
    'edad': np.where(np.random.random(n) < 0.1, np.nan,
                     np.random.normal(35, 12, n).astype(int)),
    'salario': np.where(np.random.random(n) < 0.08, np.nan,
                        np.random.lognormal(10.5, 0.5, n)),
    'departamento': np.random.choice(
        ['Ventas', 'ventas', 'VENTAS', 'IT', 'it', 'Marketing', 'RRHH', None], n),
    'fecha_ingreso': [
        f'2020-{np.random.randint(1,13):02d}-{np.random.randint(1,29):02d}'
        if np.random.random() > 0.1
        else (f'{np.random.randint(1,29)}/{np.random.randint(1,13)}/2020'
              if np.random.random() > 0.5 else None)
        for _ in range(n)
    ],
    'satisfaccion': np.random.choice(
        ['bajo', 'medio', 'alto', 'Bajo', 'ALTO', None], n),
    'evaluacion': np.where(np.random.random(n) < 0.05, np.nan,
                           np.round(np.random.uniform(1, 10, n), 1)),
    'ciudad': np.random.choice(
        ['Madrid', 'Barcelona', 'Sevilla', 'Valencia', 'madrid', 'BARCELONA'], n),
})

# Inyectar outliers
df_dirty.loc[10, 'salario'] = 500000
df_dirty.loc[20, 'edad'] = 150
df_dirty.loc[30, 'edad'] = -5

# Inyectar duplicados (15 filas duplicadas)
df_dirty = pd.concat([df_dirty, df_dirty.iloc[:15]], ignore_index=True)

print(f"Dataset sucio creado: {df_dirty.shape[0]} filas x {df_dirty.shape[1]} columnas")
print(f"Problemas inyectados: nulos, duplicados, outliers, tipos incorrectos, strings inconsistentes, fechas mixtas")
df_dirty.head(10)

## Referencia de Formulas y Metodos

<div style="background: #eaf2f8; padding: 20px; border-radius: 10px; border-left: 4px solid #3498db; margin-bottom: 15px;">

### Deteccion de Outliers

**Metodo IQR (Rango Intercuartilico):**
- `Q1 = percentil 25`, `Q3 = percentil 75`
- `IQR = Q3 - Q1`
- Limites: `[Q1 - 1.5 * IQR, Q3 + 1.5 * IQR]`
- Todo valor fuera de estos limites se considera outlier.

**Z-score:**
- `z = (x - media) / desviacion_estandar`
- Si `|z| > 3`, el valor es un outlier (a mas de 3 desviaciones estandar de la media).

</div>

<div style="background: #eafaf1; padding: 20px; border-radius: 10px; border-left: 4px solid #2ecc71; margin-bottom: 15px;">

### Escalado

**MinMaxScaler:**
- `x_scaled = (x - x_min) / (x_max - x_min)`
- Rango resultante: [0, 1]. Sensible a outliers.

**StandardScaler:**
- `x_scaled = (x - media) / desviacion_estandar`
- Resultado: media=0, desviacion=1. Sensible a outliers.

**RobustScaler:**
- `x_scaled = (x - mediana) / IQR`
- Usa mediana e IQR, por lo que es robusto a outliers.

</div>

<div style="background: #fef9e7; padding: 20px; border-radius: 10px; border-left: 4px solid #f39c12; margin-bottom: 15px;">

### Encoding Categorico

**One-Hot Encoding:**
- Crea una columna binaria (0/1) por cada categoria unica.
- `drop_first=True` evita la trampa de variables dummy (multicolinealidad).

**Ordinal Encoding:**
- Asigna un entero a cada categoria segun un orden logico (ej: bajo=1, medio=2, alto=3).

**Target Encoding:**
- Reemplaza cada categoria por la media de la variable objetivo para esa categoria.

</div>

<div style="background: #fdedec; padding: 20px; border-radius: 10px; border-left: 4px solid #e74c3c; margin-bottom: 15px;">

### Estrategias de Imputacion

| Estrategia | Cuando usarla | Formula/Metodo |
|------------|---------------|----------------|
| **Media** | Datos numericos sin outliers extremos | `valor = media(columna)` |
| **Mediana** | Datos numericos con outliers | `valor = mediana(columna)` |
| **Moda** | Datos categoricos | `valor = moda(columna)` |
| **Forward-fill** | Series temporales | `valor = ultimo valor valido anterior` |
| **Imputacion por grupo** | Cuando hay variable de agrupacion logica | `valor = media(columna, por grupo)` |
| **KNN Imputer** | Datos con estructura local | `valor = media ponderada de k vecinos mas cercanos` |

**V de Cramer** (asociacion entre categoricas):
- `V = sqrt(chi2 / (n * (min(r, c) - 1)))`
- Donde `chi2` es el estadistico chi-cuadrado, `n` el total de observaciones, `r` filas y `c` columnas de la tabla de contingencia.

</div>

**Ejercicio 1:** Perfilar el DataFrame `df_dirty`. Obtener su forma (shape), tipos de datos (dtypes), conteo de valores nulos por columna y estadisticas descriptivas (describe). Interpretar los resultados para identificar los problemas mas evidentes del dataset.

**Resolucion:**

1. `df.shape` devuelve una tupla `(filas, columnas)` que indica las dimensiones del DataFrame.
2. `df.dtypes` muestra el tipo de dato de cada columna. Permite detectar columnas que deberian ser numericas pero aparecen como `object` (texto).
3. `df.isnull().sum()` cuenta los valores nulos por columna. Se complementa con `df.isnull().mean() * 100` para obtener el porcentaje.
4. `df.describe()` genera estadisticas descriptivas (count, mean, std, min, percentiles, max) solo para columnas numericas por defecto. Con `include='all'` se incluyen tambien las categoricas.
5. `df.info()` proporciona un resumen compacto con tipos, conteo de no-nulos y uso de memoria.

In [ ]:
# --- Ejercicio 1: Perfilar df_dirty ---

print("=" * 60)
print("FORMA DEL DATASET")
print("=" * 60)
print(f"Filas: {df_dirty.shape[0]}, Columnas: {df_dirty.shape[1]}")

print("\n" + "=" * 60)
print("TIPOS DE DATOS")
print("=" * 60)
print(df_dirty.dtypes)

print("\n" + "=" * 60)
print("VALORES NULOS")
print("=" * 60)
nulls = pd.DataFrame({
    'Nulos': df_dirty.isnull().sum(),
    '% Nulos': (df_dirty.isnull().mean() * 100).round(2)
})
print(nulls)

print("\n" + "=" * 60)
print("ESTADISTICAS DESCRIPTIVAS (numericas)")
print("=" * 60)
display(df_dirty.describe())

print("\n" + "=" * 60)
print("ESTADISTICAS DESCRIPTIVAS (todas)")
print("=" * 60)
display(df_dirty.describe(include='all'))

print("\n" + "=" * 60)
print("INFO GENERAL")
print("=" * 60)
df_dirty.info()

**Ejercicio 2:** Identificar las columnas que tienen mas del 5% de valores nulos, las columnas con tipos de datos incorrectos (por ejemplo, columnas que deberian ser numericas pero son `object`) y las columnas constantes (con un solo valor unico). Presentar los resultados de forma estructurada.

**Resolucion:**

1. **Columnas con >5% nulos:** Se calcula `df.isnull().mean()` y se filtran las columnas cuyo porcentaje supera 0.05. Esto identifica donde la falta de datos es significativa y requiere estrategia de imputacion.
2. **Tipos incorrectos:** Se comparan los dtypes reales contra los esperados. Una columna como `fecha_ingreso` aparece como `object` cuando deberia ser `datetime64`. Columnas numericas con nulos en pandas se convierten a `float64` automaticamente.
3. **Columnas constantes:** Se usa `df.nunique()` y se filtran las que tienen `n_unique == 1`. Columnas constantes no aportan informacion al modelo y deben eliminarse.

In [ ]:
# --- Ejercicio 2: Identificar columnas problematicas ---

# Columnas con >5% nulos
pct_nulls = df_dirty.isnull().mean() * 100
cols_many_nulls = pct_nulls[pct_nulls > 5]
print("Columnas con >5% de valores nulos:")
print(cols_many_nulls.to_string())

# Tipos incorrectos
print("\n--- Tipos de datos actuales ---")
tipos_esperados = {
    'id': 'int', 'nombre': 'object', 'edad': 'float/int',
    'salario': 'float', 'departamento': 'object',
    'fecha_ingreso': 'datetime64', 'satisfaccion': 'object/ordinal',
    'evaluacion': 'float', 'ciudad': 'object'
}
for col in df_dirty.columns:
    actual = str(df_dirty[col].dtype)
    esperado = tipos_esperados.get(col, '?')
    marca = ' << REVISAR' if ('object' in actual and 'date' in esperado.lower()) else ''
    print(f"  {col:20s} actual={actual:10s} esperado={esperado}{marca}")

# Columnas constantes
n_unique = df_dirty.nunique()
cols_constantes = n_unique[n_unique <= 1]
print(f"\nColumnas constantes (n_unique <= 1): {list(cols_constantes.index) if len(cols_constantes) > 0 else 'Ninguna'}")

# Visualizacion resumen
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(pct_nulls.index, pct_nulls.values, color=[C_DANGER if v > 5 else C_PRIMARY for v in pct_nulls.values])
axes[0].axvline(x=5, color=C_ORANGE, linestyle='--', label='Umbral 5%')
axes[0].set_xlabel('% Nulos')
axes[0].set_title('Porcentaje de valores nulos por columna', fontsize=12, color=C_DARK)
axes[0].legend()

axes[1].barh(n_unique.index, n_unique.values, color=C_PRIMARY)
axes[1].set_xlabel('Valores unicos')
axes[1].set_title('Numero de valores unicos por columna', fontsize=12, color=C_DARK)

plt.tight_layout()
plt.show()

**Ejercicio 3:** Generar un reporte de calidad de datos en forma de tabla que muestre, para cada columna: nombre, tipo de dato, numero de valores nulos, porcentaje de nulos, numero de valores unicos y un ejemplo de valor. Ordenar por porcentaje de nulos de mayor a menor.

**Resolucion:**

1. Se itera sobre cada columna del DataFrame para recopilar sus metadatos.
2. `df[col].dtype` obtiene el tipo de dato.
3. `df[col].isnull().sum()` cuenta nulos; dividido entre `len(df)` y multiplicado por 100 da el porcentaje.
4. `df[col].nunique()` cuenta valores unicos (excluyendo nulos).
5. `df[col].dropna().iloc[0]` obtiene el primer valor no nulo como ejemplo.
6. Se construye un DataFrame con toda esta informacion y se ordena con `sort_values('% Nulos', ascending=False)`.
7. Se aplica estilo con `Styler.background_gradient()` para resaltar visualmente las columnas mas problematicas.

In [ ]:
# --- Ejercicio 3: Reporte de calidad de datos ---

reporte = []
for col in df_dirty.columns:
    n_nulls = df_dirty[col].isnull().sum()
    pct_nulls = round(n_nulls / len(df_dirty) * 100, 2)
    n_unique = df_dirty[col].nunique()
    ejemplo = df_dirty[col].dropna().iloc[0] if df_dirty[col].notna().any() else 'N/A'
    reporte.append({
        'Columna': col,
        'Tipo': str(df_dirty[col].dtype),
        'Nulos': n_nulls,
        '% Nulos': pct_nulls,
        'Unicos': n_unique,
        'Ejemplo': ejemplo
    })

df_reporte = pd.DataFrame(reporte).sort_values('% Nulos', ascending=False).reset_index(drop=True)

# Mostrar con estilo
styled = df_reporte.style.background_gradient(
    subset=['% Nulos'], cmap='Reds'
).background_gradient(
    subset=['Unicos'], cmap='Blues'
).set_caption('Reporte de Calidad de Datos')

display(styled)

**Ejercicio 4:** Crear un heatmap de valores faltantes (missing values) del DataFrame `df_dirty`. Utilizar seaborn o matplotlib para visualizar la matriz de nulidad, donde cada fila es una observacion y cada columna es una variable, coloreando en un tono distinto las celdas con datos faltantes.

**Resolucion:**

1. Se crea una matriz booleana con `df.isnull()` donde `True` indica dato faltante y `False` dato presente.
2. Se convierte a enteros (0 y 1) para poder usar `sns.heatmap()`.
3. `sns.heatmap()` recibe los parametros:
   - `data`: la matriz booleana convertida a int.
   - `cbar_kws`: personaliza la barra de color.
   - `cmap`: mapa de colores; se usa un mapa de 2 colores (presente vs faltante).
   - `yticklabels=False`: oculta las etiquetas del eje Y (demasiadas filas).
4. La visualizacion permite detectar patrones de nulidad: si los nulos estan dispersos aleatoriamente (MCAR), concentrados en ciertas filas (MAR) o sistematicos (MNAR).

In [ ]:
# --- Ejercicio 4: Heatmap de missing values ---

from matplotlib.colors import ListedColormap

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap de nulidad
cmap_missing = ListedColormap([C_PRIMARY, C_DANGER])
sns.heatmap(df_dirty.isnull().astype(int), cmap=cmap_missing, cbar_kws={'label': '0=Presente, 1=Faltante'},
            yticklabels=False, ax=axes[0])
axes[0].set_title('Matriz de valores faltantes', fontsize=13, color=C_DARK)
axes[0].set_xlabel('Columnas')
axes[0].set_ylabel('Observaciones')

# Barplot de porcentaje de nulos
pct_null = df_dirty.isnull().mean() * 100
colors = [C_DANGER if v > 5 else C_SUCCESS for v in pct_null.values]
axes[1].barh(pct_null.index, pct_null.values, color=colors)
axes[1].axvline(x=5, color=C_ORANGE, linestyle='--', linewidth=2, label='Umbral 5%')
axes[1].set_xlabel('% Valores faltantes')
axes[1].set_title('Porcentaje de nulos por columna', fontsize=13, color=C_DARK)
axes[1].legend()

plt.tight_layout()
plt.show()

**Ejercicio 5:** Imputar los valores nulos de la columna `edad` de dos formas: con la media y con la mediana. Comparar ambas estrategias visualmente mediante histogramas antes y despues de la imputacion. Discutir cual es mas apropiada cuando existen outliers.

**Resolucion:**

1. **Media:** `valor_imputado = media(columna)`. La media es sensible a outliers; si hay valores extremos como edad=150, la media se desplaza hacia arriba.
2. **Mediana:** `valor_imputado = mediana(columna)`. La mediana es el valor central de los datos ordenados; es robusta a outliers porque no se ve afectada por valores extremos.
3. Se usa `df['edad'].fillna(valor)` para reemplazar los NaN por el valor calculado.
4. Se comparan tres histogramas: datos originales (con NaN excluidos), imputados con media, e imputados con mediana.
5. **Conclusion:** Cuando hay outliers, la mediana preserva mejor la distribucion original porque no se desplaza hacia los extremos.

In [ ]:
# --- Ejercicio 5: Imputar edad con media vs mediana ---

media_edad = df_dirty['edad'].mean()
mediana_edad = df_dirty['edad'].median()

edad_media = df_dirty['edad'].fillna(media_edad)
edad_mediana = df_dirty['edad'].fillna(mediana_edad)

print(f"Nulos originales en 'edad': {df_dirty['edad'].isnull().sum()}")
print(f"Media de edad (con outliers): {media_edad:.2f}")
print(f"Mediana de edad: {mediana_edad:.2f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df_dirty['edad'].dropna(), bins=30, color=C_PRIMARY, alpha=0.7, edgecolor='white')
axes[0].axvline(media_edad, color=C_DANGER, linestyle='--', label=f'Media={media_edad:.1f}')
axes[0].axvline(mediana_edad, color=C_SUCCESS, linestyle='--', label=f'Mediana={mediana_edad:.1f}')
axes[0].set_title('ANTES (sin imputar)', fontsize=12, color=C_DARK)
axes[0].set_xlabel('Edad')
axes[0].legend()

axes[1].hist(edad_media, bins=30, color=C_DANGER, alpha=0.7, edgecolor='white')
axes[1].axvline(media_edad, color=C_DARK, linestyle='--', label=f'Media={media_edad:.1f}')
axes[1].set_title('DESPUES: imputado con MEDIA', fontsize=12, color=C_DANGER)
axes[1].set_xlabel('Edad')
axes[1].legend()

axes[2].hist(edad_mediana, bins=30, color=C_SUCCESS, alpha=0.7, edgecolor='white')
axes[2].axvline(mediana_edad, color=C_DARK, linestyle='--', label=f'Mediana={mediana_edad:.1f}')
axes[2].set_title('DESPUES: imputado con MEDIANA', fontsize=12, color=C_SUCCESS)
axes[2].set_xlabel('Edad')
axes[2].legend()

plt.suptitle('Comparacion: Imputacion con Media vs Mediana en "edad"', fontsize=14, color=C_DARK, y=1.02)
plt.tight_layout()
plt.show()

**Ejercicio 6:** Imputar los valores nulos de la columna `departamento` (categorica) utilizando la moda (valor mas frecuente). Mostrar la distribucion de valores antes y despues de la imputacion con un grafico de barras.

**Resolucion:**

1. **Moda:** `df['columna'].mode()[0]` devuelve el valor mas frecuente de la columna. Se usa `[0]` porque `mode()` puede devolver multiples valores si hay empate.
2. `df['columna'].fillna(moda)` reemplaza todos los NaN por la moda.
3. Para variables categoricas, la moda es la unica medida de tendencia central aplicable (no se puede calcular media ni mediana de texto).
4. Se comparan las distribuciones con graficos de barras lado a lado para verificar que la imputacion no distorsione excesivamente las proporciones originales.
5. **Nota:** En este dataset, `departamento` tiene inconsistencias de case (Ventas, ventas, VENTAS) que se tratan en ejercicios posteriores.

In [ ]:
# --- Ejercicio 6: Imputar departamento con moda ---

moda_depto = df_dirty['departamento'].mode()[0]
print(f"Moda de 'departamento': {moda_depto}")
print(f"Nulos antes: {df_dirty['departamento'].isnull().sum()}")

depto_imputado = df_dirty['departamento'].fillna(moda_depto)
print(f"Nulos despues: {depto_imputado.isnull().sum()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Antes
conteo_antes = df_dirty['departamento'].value_counts(dropna=False)
conteo_antes.plot(kind='bar', ax=axes[0], color=C_PRIMARY, edgecolor='white')
axes[0].set_title('ANTES: distribucion de departamento', fontsize=12, color=C_DARK)
axes[0].set_xlabel('Departamento')
axes[0].set_ylabel('Frecuencia')
axes[0].tick_params(axis='x', rotation=45)

# Despues
conteo_despues = depto_imputado.value_counts(dropna=False)
colores = [C_SUCCESS if idx == moda_depto else C_PRIMARY for idx in conteo_despues.index]
conteo_despues.plot(kind='bar', ax=axes[1], color=colores, edgecolor='white')
axes[1].set_title('DESPUES: imputado con moda', fontsize=12, color=C_SUCCESS)
axes[1].set_xlabel('Departamento')
axes[1].set_ylabel('Frecuencia')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

**Ejercicio 7:** Crear una serie temporal simulada con valores faltantes e imputar utilizando forward-fill (propagacion hacia adelante). Visualizar la serie antes y despues de la imputacion para verificar que los huecos se rellenan con el ultimo valor valido.

**Resolucion:**

1. Se genera una serie temporal con `pd.date_range()` y valores sinteticos (por ejemplo, seno + ruido).
2. Se inyectan NaN en posiciones aleatorias para simular datos faltantes.
3. **Forward-fill** (`df.ffill()` o `df.fillna(method='ffill')`): propaga el ultimo valor valido hacia adelante. Es decir, cada NaN se reemplaza por el valor inmediatamente anterior que no sea NaN.
4. Es apropiado para series temporales porque asume que el valor mas reciente es la mejor estimacion cuando no hay medicion.
5. **Limitacion:** Si el primer valor es NaN, forward-fill no puede rellenarlo (no hay valor anterior). En ese caso se complementa con backward-fill (`bfill`).

In [ ]:
# --- Ejercicio 7: Forward-fill en serie temporal ---

np.random.seed(42)
fechas = pd.date_range('2023-01-01', periods=100, freq='D')
valores = np.sin(np.linspace(0, 4 * np.pi, 100)) * 50 + 100 + np.random.normal(0, 5, 100)

ts = pd.Series(valores, index=fechas, name='ventas')

# Inyectar NaN en posiciones aleatorias
idx_nan = np.random.choice(range(1, 100), size=20, replace=False)
ts_con_nan = ts.copy()
ts_con_nan.iloc[idx_nan] = np.nan

# Forward-fill
ts_ffill = ts_con_nan.ffill()

print(f"Nulos antes: {ts_con_nan.isnull().sum()}")
print(f"Nulos despues de ffill: {ts_ffill.isnull().sum()}")

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# Antes
axes[0].plot(ts_con_nan.index, ts_con_nan.values, color=C_PRIMARY, marker='o', markersize=3, label='Datos presentes')
mask_nan = ts_con_nan.isnull()
axes[0].scatter(ts_con_nan.index[mask_nan], [ts.iloc[i] for i in range(len(ts)) if mask_nan.iloc[i]],
                color=C_DANGER, marker='x', s=80, zorder=5, label='Valores faltantes (posicion real)')
axes[0].set_title('ANTES: serie temporal con valores faltantes', fontsize=12, color=C_DARK)
axes[0].legend()
axes[0].set_ylabel('Ventas')

# Despues
axes[1].plot(ts_ffill.index, ts_ffill.values, color=C_SUCCESS, marker='o', markersize=3, label='Datos con ffill')
axes[1].scatter(ts_con_nan.index[mask_nan], ts_ffill[mask_nan],
                color=C_ORANGE, marker='s', s=60, zorder=5, label='Valores imputados (ffill)')
axes[1].set_title('DESPUES: imputado con forward-fill', fontsize=12, color=C_SUCCESS)
axes[1].legend()
axes[1].set_ylabel('Ventas')
axes[1].set_xlabel('Fecha')

plt.tight_layout()
plt.show()

**Ejercicio 8:** Imputar los valores nulos de `salario` utilizando la media de salario por departamento (imputacion por grupo). Primero normalizar `departamento` a title case, luego calcular la media de salario para cada departamento y rellenar los nulos con la media correspondiente a su grupo.

**Resolucion:**

1. **Normalizar departamento:** `df['departamento'].str.strip().str.title()` unifica las variantes (Ventas, ventas, VENTAS -> Ventas).
2. **Calcular media por grupo:** `df.groupby('depto_norm')['salario'].transform('mean')` calcula la media de salario para cada departamento y la asigna a cada fila del grupo.
3. **Imputar:** `df['salario'].fillna(media_por_grupo)` rellena cada NaN con la media de su departamento.
4. Esta estrategia es superior a la media global porque respeta la estructura de los datos: si el departamento de IT tiene salarios mas altos que RRHH, un nulo en IT se rellena con la media de IT, no con la media general.
5. Si el departamento tambien es NaN, se usa la media global como fallback.

In [ ]:
# --- Ejercicio 8: Imputar salario por grupo (departamento) ---

df_temp = df_dirty.copy()
df_temp['depto_norm'] = df_temp['departamento'].str.strip().str.title()

# Media por departamento
media_por_depto = df_temp.groupby('depto_norm')['salario'].mean()
print("Media de salario por departamento:")
print(media_por_depto.round(2).to_string())

# Imputar por grupo
media_grupo = df_temp.groupby('depto_norm')['salario'].transform('mean')
salario_antes = df_temp['salario'].copy()
df_temp['salario_imputado'] = df_temp['salario'].fillna(media_grupo)
# Fallback con media global si departamento es NaN
df_temp['salario_imputado'] = df_temp['salario_imputado'].fillna(df_temp['salario'].mean())

print(f"\nNulos antes: {salario_antes.isnull().sum()}")
print(f"Nulos despues: {df_temp['salario_imputado'].isnull().sum()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(salario_antes.dropna(), bins=40, color=C_PRIMARY, alpha=0.7, edgecolor='white')
axes[0].set_title('ANTES: distribucion de salario', fontsize=12, color=C_DARK)
axes[0].set_xlabel('Salario')

axes[1].hist(df_temp['salario_imputado'], bins=40, color=C_SUCCESS, alpha=0.7, edgecolor='white')
axes[1].set_title('DESPUES: imputado por media de departamento', fontsize=12, color=C_SUCCESS)
axes[1].set_xlabel('Salario')

plt.tight_layout()
plt.show()

del df_temp

**Ejercicio 9:** Utilizar `KNNImputer` de scikit-learn para imputar las columnas numericas (`edad`, `salario`, `evaluacion`) y comparar el resultado con la imputacion por media simple. Visualizar las distribuciones resultantes de ambos metodos.

**Resolucion:**

1. **KNNImputer:** Imputa cada valor faltante utilizando la media ponderada de los `n_neighbors` vecinos mas cercanos (por defecto k=5). La distancia se calcula en el espacio de las columnas numericas.
   - Parametro `n_neighbors`: numero de vecinos a considerar (default=5).
   - Parametro `weights`: 'uniform' (todos pesan igual) o 'distance' (los mas cercanos pesan mas).
2. **SimpleImputer(strategy='mean'):** Reemplaza cada NaN por la media de la columna, sin considerar la estructura de los datos.
3. KNNImputer es mas sofisticado porque tiene en cuenta las relaciones entre variables: si una persona tiene alta evaluacion y esta en IT, su salario imputado reflejara esas caracteristicas.
4. Se comparan ambos metodos con histogramas para cada columna numerica.

In [ ]:
# --- Ejercicio 9: KNN Imputer vs Media ---

cols_num = ['edad', 'salario', 'evaluacion']
df_num = df_dirty[cols_num].copy()

# Imputacion con media
imp_media = SimpleImputer(strategy='mean')
df_media = pd.DataFrame(imp_media.fit_transform(df_num), columns=cols_num)

# Imputacion con KNN
imp_knn = KNNImputer(n_neighbors=5, weights='distance')
df_knn = pd.DataFrame(imp_knn.fit_transform(df_num), columns=cols_num)

fig, axes = plt.subplots(len(cols_num), 3, figsize=(18, 5 * len(cols_num)))

for i, col in enumerate(cols_num):
    # Original
    axes[i, 0].hist(df_num[col].dropna(), bins=30, color=C_PRIMARY, alpha=0.7, edgecolor='white')
    axes[i, 0].set_title(f'Original: {col} (sin NaN)', fontsize=11, color=C_DARK)
    axes[i, 0].set_xlabel(col)

    # Media
    axes[i, 1].hist(df_media[col], bins=30, color=C_DANGER, alpha=0.7, edgecolor='white')
    axes[i, 1].set_title(f'Imputado con MEDIA: {col}', fontsize=11, color=C_DANGER)
    axes[i, 1].set_xlabel(col)

    # KNN
    axes[i, 2].hist(df_knn[col], bins=30, color=C_SUCCESS, alpha=0.7, edgecolor='white')
    axes[i, 2].set_title(f'Imputado con KNN: {col}', fontsize=11, color=C_SUCCESS)
    axes[i, 2].set_xlabel(col)

plt.suptitle('Comparacion: Imputacion con Media vs KNN Imputer', fontsize=14, color=C_DARK, y=1.01)
plt.tight_layout()
plt.show()

# Estadisticas comparativas
print("\n--- Estadisticas comparativas ---")
for col in cols_num:
    print(f"\n{col}:")
    print(f"  Original  -> media={df_num[col].mean():.2f}, std={df_num[col].std():.2f}")
    print(f"  Imp.Media -> media={df_media[col].mean():.2f}, std={df_media[col].std():.2f}")
    print(f"  Imp.KNN   -> media={df_knn[col].mean():.2f}, std={df_knn[col].std():.2f}")

**Ejercicio 10:** Antes de imputar `salario`, crear una columna binaria `salario_missing` que valga 1 donde `salario` es NaN y 0 donde tiene valor. Luego imputar `salario` con la mediana. Mostrar la tabla con ambas columnas y verificar que la flag se creo correctamente.

**Resolucion:**

1. Se crea la columna flag con `df['salario_missing'] = df['salario'].isnull().astype(int)`. Esto convierte True/False a 1/0.
2. La flag se crea **antes** de imputar, ya que despues de la imputacion ya no se sabra cuales valores eran originalmente faltantes.
3. Esta tecnica es util porque la "razon de la ausencia" puede contener informacion predictiva. Por ejemplo, si los salarios faltantes corresponden sistematicamente a empleados nuevos, el modelo puede aprender ese patron.
4. Despues se imputa con la mediana: `df['salario'].fillna(df['salario'].median())`.
5. Se verifica la coherencia: donde `salario_missing == 1`, el valor de `salario` despues de imputar debe ser igual a la mediana.

In [ ]:
# --- Ejercicio 10: Flag de missing antes de imputar ---

df_temp = df_dirty[['id', 'salario']].copy()

# Paso 1: crear flag ANTES de imputar
df_temp['salario_missing'] = df_temp['salario'].isnull().astype(int)

# Paso 2: imputar con mediana
mediana_salario = df_temp['salario'].median()
df_temp['salario_imputado'] = df_temp['salario'].fillna(mediana_salario)

print(f"Mediana de salario: {mediana_salario:.2f}")
print(f"Total filas con salario faltante: {df_temp['salario_missing'].sum()}")
print(f"Nulos en salario_imputado: {df_temp['salario_imputado'].isnull().sum()}")

# Verificar: donde flag=1, el valor imputado debe ser la mediana
verificacion = df_temp[df_temp['salario_missing'] == 1][['salario', 'salario_missing', 'salario_imputado']]
print(f"\nFilas con flag=1 (muestra):")
display(verificacion.head(10))

# Visualizacion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_dirty['salario'].dropna(), bins=40, color=C_PRIMARY, alpha=0.7, edgecolor='white', label='Original')
axes[0].axvline(mediana_salario, color=C_DANGER, linestyle='--', label=f'Mediana={mediana_salario:.0f}')
axes[0].set_title('ANTES: salario con NaN', fontsize=12, color=C_DARK)
axes[0].legend()

axes[1].hist(df_temp['salario_imputado'], bins=40, color=C_SUCCESS, alpha=0.7, edgecolor='white', label='Imputado')
axes[1].axvline(mediana_salario, color=C_DANGER, linestyle='--', label=f'Mediana={mediana_salario:.0f}')
axes[1].set_title('DESPUES: salario imputado + flag creada', fontsize=12, color=C_SUCCESS)
axes[1].legend()

plt.tight_layout()
plt.show()

del df_temp

**Ejercicio 11:** Detectar y eliminar los duplicados exactos en `df_dirty`. Mostrar cuantos duplicados hay, cuales son las filas duplicadas, y el shape del DataFrame antes y despues de la eliminacion.

**Resolucion:**

1. `df.duplicated()` devuelve una serie booleana donde `True` indica que la fila es duplicada (tiene una fila identica anterior).
2. `df.duplicated().sum()` cuenta el total de duplicados.
3. `df[df.duplicated(keep=False)]` muestra todas las filas involucradas en duplicacion (tanto la primera aparicion como las copias).
4. `df.drop_duplicates()` elimina las filas duplicadas, conservando la primera aparicion por defecto.
   - `keep='first'`: conserva la primera aparicion (default).
   - `keep='last'`: conserva la ultima aparicion.
   - `keep=False`: elimina todas las apariciones.
5. Se verifica comparando `df.shape` antes y despues.

In [ ]:
# --- Ejercicio 11: Detectar y eliminar duplicados exactos ---

n_dupes = df_dirty.duplicated().sum()
print(f"ANTES: {df_dirty.shape[0]} filas, {n_dupes} duplicados exactos detectados")

# Mostrar las filas duplicadas
print("\nFilas duplicadas (muestra):")
display(df_dirty[df_dirty.duplicated(keep=False)].head(10))

# Eliminar duplicados
df_sin_dupes = df_dirty.drop_duplicates(keep='first')
print(f"\nDESPUES: {df_sin_dupes.shape[0]} filas (se eliminaron {df_dirty.shape[0] - df_sin_dupes.shape[0]} filas)")

# Visualizacion
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(['Antes', 'Despues'], [df_dirty.shape[0], df_sin_dupes.shape[0]],
              color=[C_DANGER, C_SUCCESS], edgecolor='white', width=0.5)
ax.bar_label(bars, fontsize=12, fontweight='bold')
ax.set_ylabel('Numero de filas')
ax.set_title('Eliminacion de duplicados exactos', fontsize=13, color=C_DARK)
plt.tight_layout()
plt.show()

**Ejercicio 12:** Detectar duplicados aproximados en la columna `nombre`. Normalizar los nombres (strip, lower, quitar espacios multiples) y luego identificar cuales nombres, tras la normalizacion, resultan ser el mismo. Mostrar los grupos de nombres que son duplicados aproximados.

**Resolucion:**

1. **Normalizar:** Se aplica una cadena de transformaciones:
   - `str.strip()`: elimina espacios al inicio y final.
   - `str.lower()`: convierte a minusculas para que "MARIA LOPEZ" y "maria lopez" sean iguales.
   - `str.replace(r'\s+', ' ', regex=True)`: reemplaza multiples espacios por uno solo.
2. Se compara el numero de valores unicos antes y despues de la normalizacion para cuantificar cuantos duplicados aproximados habia.
3. Se agrupan los nombres originales por su version normalizada para ver las variantes que representan la misma persona.
4. Esta tecnica es fundamental en datos del mundo real donde los nombres se capturan con diferentes formatos.

In [ ]:
# --- Ejercicio 12: Duplicados aproximados en nombre ---

import re

df_temp = df_dirty[['nombre']].dropna().copy()
df_temp['nombre_original'] = df_temp['nombre']
df_temp['nombre_norm'] = (df_temp['nombre']
                          .str.strip()
                          .str.lower()
                          .str.replace(r'\s+', ' ', regex=True))

n_antes = df_temp['nombre_original'].nunique()
n_despues = df_temp['nombre_norm'].nunique()
print(f"Valores unicos ANTES de normalizar: {n_antes}")
print(f"Valores unicos DESPUES de normalizar: {n_despues}")
print(f"Duplicados aproximados detectados: {n_antes - n_despues}")

# Mostrar grupos de variantes
print("\n--- Variantes por nombre normalizado ---")
grupos = df_temp.groupby('nombre_norm')['nombre_original'].unique()
for norm, variantes in grupos.items():
    if len(variantes) > 1:
        print(f"  '{norm}' <- variantes: {list(variantes)}")

# Visualizacion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_temp['nombre_original'].value_counts().plot(kind='bar', ax=axes[0], color=C_DANGER, edgecolor='white')
axes[0].set_title('ANTES: conteo por nombre (sin normalizar)', fontsize=11, color=C_DARK)
axes[0].tick_params(axis='x', rotation=45)

df_temp['nombre_norm'].value_counts().plot(kind='bar', ax=axes[1], color=C_SUCCESS, edgecolor='white')
axes[1].set_title('DESPUES: conteo por nombre normalizado', fontsize=11, color=C_SUCCESS)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

del df_temp

**Ejercicio 13:** Detectar duplicados basados en el subconjunto de columnas `['nombre', 'departamento']`, conservando solo la primera aparicion (`keep='first'`). Mostrar cuantos registros se eliminan y una muestra de los duplicados encontrados.

**Resolucion:**

1. `df.duplicated(subset=['nombre', 'departamento'])` marca como duplicada toda fila cuya combinacion de nombre y departamento ya aparecio antes.
2. A diferencia de los duplicados exactos (todas las columnas), aqui se buscan filas que comparten las mismas claves de negocio aunque difieran en otras columnas.
3. `keep='first'` conserva la primera aparicion y marca las demas como duplicadas.
4. `df.drop_duplicates(subset=['nombre', 'departamento'], keep='first')` elimina las marcadas.
5. Este tipo de deduplicacion es comun cuando se fusionan bases de datos de diferentes fuentes.

In [ ]:
# --- Ejercicio 13: Duplicados por subconjunto ['nombre', 'departamento'] ---

subset_cols = ['nombre', 'departamento']
n_dupes_subset = df_dirty.duplicated(subset=subset_cols, keep='first').sum()
print(f"Duplicados por subconjunto {subset_cols}: {n_dupes_subset}")

# Mostrar muestra de duplicados
dupes_mask = df_dirty.duplicated(subset=subset_cols, keep=False)
print(f"\nFilas involucradas en duplicacion (muestra):")
display(df_dirty[dupes_mask].sort_values(subset_cols).head(15))

# Eliminar
df_dedup = df_dirty.drop_duplicates(subset=subset_cols, keep='first')
print(f"\nAntes: {df_dirty.shape[0]} filas")
print(f"Despues: {df_dedup.shape[0]} filas")
print(f"Eliminadas: {df_dirty.shape[0] - df_dedup.shape[0]}")

# Visualizacion
fig, ax = plt.subplots(figsize=(8, 4))
categorias = ['Antes', 'Despues']
valores = [df_dirty.shape[0], df_dedup.shape[0]]
bars = ax.bar(categorias, valores, color=[C_DANGER, C_SUCCESS], edgecolor='white', width=0.5)
ax.bar_label(bars, fontsize=12, fontweight='bold')
ax.set_ylabel('Numero de filas')
ax.set_title(f'Deduplicacion por subconjunto {subset_cols}', fontsize=13, color=C_DARK)
plt.tight_layout()
plt.show()

**Ejercicio 14:** Detectar outliers en la columna `salario` utilizando el metodo IQR (Rango Intercuartilico). Calcular Q1, Q3, IQR, y los limites inferior y superior. Identificar los outliers y visualizarlos con un boxplot antes y despues de su eliminacion.

**Resolucion:**

1. **Q1** = percentil 25 de la columna: `df['salario'].quantile(0.25)`.
2. **Q3** = percentil 75 de la columna: `df['salario'].quantile(0.75)`.
3. **IQR** = Q3 - Q1: rango intercuartilico, mide la dispersion del 50% central de los datos.
4. **Limite inferior** = Q1 - 1.5 * IQR: valores por debajo son outliers.
5. **Limite superior** = Q3 + 1.5 * IQR: valores por encima son outliers.
6. Un outlier es cualquier valor `x` tal que `x < limite_inferior` o `x > limite_superior`.
7. Se usa un boxplot que automaticamente muestra los bigotes (limites) y los puntos fuera de ellos (outliers).

In [ ]:
# --- Ejercicio 14: Outliers en salario con IQR ---

salario = df_dirty['salario'].dropna()

Q1 = salario.quantile(0.25)
Q3 = salario.quantile(0.75)
IQR = Q3 - Q1
lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR

outliers_mask = (salario < lim_inf) | (salario > lim_sup)
n_outliers = outliers_mask.sum()

print(f"Q1 = {Q1:.2f}")
print(f"Q3 = {Q3:.2f}")
print(f"IQR = {IQR:.2f}")
print(f"Limite inferior = {lim_inf:.2f}")
print(f"Limite superior = {lim_sup:.2f}")
print(f"Outliers detectados: {n_outliers} ({n_outliers/len(salario)*100:.1f}%)")
print(f"\nValores outlier:")
print(salario[outliers_mask].values)

# Salario sin outliers
salario_clean = salario[~outliers_mask]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bp1 = axes[0].boxplot(salario.values, vert=True, patch_artist=True,
                       boxprops=dict(facecolor=C_PRIMARY, alpha=0.7),
                       flierprops=dict(marker='o', markerfacecolor=C_DANGER, markersize=6))
axes[0].set_title(f'ANTES: salario con outliers (n={len(salario)})', fontsize=12, color=C_DARK)
axes[0].set_ylabel('Salario')
axes[0].axhline(lim_sup, color=C_ORANGE, linestyle='--', label=f'Lim sup={lim_sup:.0f}')
axes[0].axhline(lim_inf, color=C_ORANGE, linestyle='--', label=f'Lim inf={lim_inf:.0f}')
axes[0].legend(fontsize=9)

bp2 = axes[1].boxplot(salario_clean.values, vert=True, patch_artist=True,
                       boxprops=dict(facecolor=C_SUCCESS, alpha=0.7))
axes[1].set_title(f'DESPUES: salario sin outliers (n={len(salario_clean)})', fontsize=12, color=C_SUCCESS)
axes[1].set_ylabel('Salario')

plt.tight_layout()
plt.show()

**Ejercicio 15:** Detectar outliers en la columna `edad` utilizando el Z-score. Calcular el Z-score de cada valor y marcar como outlier aquellos con |z| > 3. Mostrar los valores detectados y su Z-score correspondiente.

**Resolucion:**

1. **Z-score:** `z = (x - media) / desviacion_estandar`. Mide cuantas desviaciones estandar un valor esta alejado de la media.
2. Se calcula con `scipy.stats.zscore()` o manualmente con `(df['edad'] - df['edad'].mean()) / df['edad'].std()`.
3. Un valor con `|z| > 3` esta a mas de 3 desviaciones estandar de la media, lo cual es extremadamente improbable en una distribucion normal (probabilidad < 0.3%).
4. En nuestro dataset, la edad=150 y edad=-5 deberian tener Z-scores muy altos en valor absoluto.
5. Se visualiza con un scatter plot donde el eje X es el indice y el eje Y el Z-score, marcando los outliers en rojo.

In [ ]:
# --- Ejercicio 15: Outliers en edad con Z-score ---

edad = df_dirty['edad'].dropna()
z_scores = (edad - edad.mean()) / edad.std()

outliers_z = np.abs(z_scores) > 3
n_out = outliers_z.sum()

print(f"Media de edad: {edad.mean():.2f}")
print(f"Desviacion estandar: {edad.std():.2f}")
print(f"Outliers detectados (|z| > 3): {n_out}")
print(f"\nDetalle de outliers:")
for idx in edad[outliers_z].index:
    print(f"  Indice {idx}: edad={edad[idx]:.0f}, z-score={z_scores[idx]:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter de Z-scores
colors_z = [C_DANGER if abs(z) > 3 else C_PRIMARY for z in z_scores]
axes[0].scatter(range(len(z_scores)), z_scores.values, c=colors_z, alpha=0.5, s=15)
axes[0].axhline(y=3, color=C_DANGER, linestyle='--', label='z = +3')
axes[0].axhline(y=-3, color=C_DANGER, linestyle='--', label='z = -3')
axes[0].set_xlabel('Indice')
axes[0].set_ylabel('Z-score')
axes[0].set_title('Z-scores de edad (outliers en rojo)', fontsize=12, color=C_DARK)
axes[0].legend()

# Histograma antes vs despues
axes[1].hist(edad, bins=30, color=C_DANGER, alpha=0.5, label=f'Con outliers (n={len(edad)})', edgecolor='white')
axes[1].hist(edad[~outliers_z], bins=30, color=C_SUCCESS, alpha=0.7, label=f'Sin outliers (n={len(edad)-n_out})', edgecolor='white')
axes[1].set_xlabel('Edad')
axes[1].set_title('Distribucion de edad: antes vs despues', fontsize=12, color=C_DARK)
axes[1].legend()

plt.tight_layout()
plt.show()

**Ejercicio 16:** Comparar dos estrategias para tratar outliers en `salario`: (a) winsorizar al percentil 1-99 (recortar los extremos) y (b) eliminar los outliers. Visualizar las distribuciones resultantes de ambos metodos lado a lado.

**Resolucion:**

1. **Winsorizar:** Recorta los valores extremos al percentil especificado. `scipy.stats.mstats.winsorize(data, limits=[0.01, 0.01])` recorta el 1% inferior y superior.
   - Los valores por debajo del percentil 1 se reemplazan por el valor del percentil 1.
   - Los valores por encima del percentil 99 se reemplazan por el valor del percentil 99.
   - Ventaja: no se pierden filas, se preserva el tamano del dataset.
2. **Eliminar:** Se calculan los limites con IQR y se descartan las filas fuera del rango.
   - Ventaja: no se distorsionan los valores.
   - Desventaja: se pierden observaciones.
3. Se comparan ambas estrategias visualmente para decidir cual es mas apropiada segun el contexto.

In [ ]:
# --- Ejercicio 16: Winsorizar vs Eliminar outliers en salario ---

from scipy.stats.mstats import winsorize

salario = df_dirty['salario'].dropna()

# Metodo A: Winsorizar al percentil 1-99
salario_wins = pd.Series(winsorize(salario, limits=[0.01, 0.01]))

# Metodo B: Eliminar con IQR
Q1 = salario.quantile(0.25)
Q3 = salario.quantile(0.75)
IQR = Q3 - Q1
mask_ok = (salario >= Q1 - 1.5 * IQR) & (salario <= Q3 + 1.5 * IQR)
salario_elim = salario[mask_ok]

print(f"Original:     n={len(salario)}, media={salario.mean():.0f}, std={salario.std():.0f}")
print(f"Winsorizado:  n={len(salario_wins)}, media={salario_wins.mean():.0f}, std={salario_wins.std():.0f}")
print(f"Eliminados:   n={len(salario_elim)}, media={salario_elim.mean():.0f}, std={salario_elim.std():.0f}")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(salario, bins=40, color=C_PRIMARY, alpha=0.7, edgecolor='white')
axes[0].set_title(f'Original (n={len(salario)})', fontsize=12, color=C_DARK)
axes[0].set_xlabel('Salario')

axes[1].hist(salario_wins, bins=40, color=C_ORANGE, alpha=0.7, edgecolor='white')
axes[1].set_title(f'Winsorizado P1-P99 (n={len(salario_wins)})', fontsize=12, color=C_ORANGE)
axes[1].set_xlabel('Salario')

axes[2].hist(salario_elim, bins=40, color=C_SUCCESS, alpha=0.7, edgecolor='white')
axes[2].set_title(f'Outliers eliminados (n={len(salario_elim)})', fontsize=12, color=C_SUCCESS)
axes[2].set_xlabel('Salario')

plt.suptitle('Comparacion: Winsorizar vs Eliminar outliers en salario', fontsize=14, color=C_DARK, y=1.02)
plt.tight_layout()
plt.show()

**Ejercicio 17:** Utilizar Isolation Forest de scikit-learn para detectar anomalias multivariantes en las columnas `['edad', 'salario', 'evaluacion']`. Visualizar los puntos anomalos en un scatter plot de edad vs salario, coloreando diferente los puntos normales y los anomalos.

**Resolucion:**

1. **Isolation Forest:** Algoritmo basado en arboles que aisle puntos anomalos. La idea es que los outliers requieren menos particiones (cortes) para ser aislados del resto.
2. Parametros principales:
   - `contamination`: proporcion esperada de outliers en el dataset (ej: 0.05 = 5%).
   - `n_estimators`: numero de arboles (default=100).
   - `random_state`: semilla para reproducibilidad.
3. `model.fit_predict(X)` devuelve 1 para puntos normales y -1 para anomalias.
4. A diferencia de IQR y Z-score (univariantes), Isolation Forest detecta anomalias **multivariantes**: un punto puede tener edad y salario normales por separado, pero su combinacion ser anomala.
5. Se requiere eliminar NaN antes de usar Isolation Forest.

In [ ]:
# --- Ejercicio 17: Isolation Forest ---

cols_if = ['edad', 'salario', 'evaluacion']
df_if = df_dirty[cols_if].dropna().copy()

iso_forest = IsolationForest(contamination=0.05, n_estimators=100, random_state=42)
df_if['anomalia'] = iso_forest.fit_predict(df_if[cols_if])

n_anomalias = (df_if['anomalia'] == -1).sum()
print(f"Puntos analizados: {len(df_if)}")
print(f"Anomalias detectadas: {n_anomalias} ({n_anomalias/len(df_if)*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter edad vs salario
normales = df_if[df_if['anomalia'] == 1]
anomalos = df_if[df_if['anomalia'] == -1]

axes[0].scatter(normales['edad'], normales['salario'], c=C_PRIMARY, alpha=0.4, s=20, label=f'Normal (n={len(normales)})')
axes[0].scatter(anomalos['edad'], anomalos['salario'], c=C_DANGER, alpha=0.8, s=60, marker='X', label=f'Anomalia (n={len(anomalos)})')
axes[0].set_xlabel('Edad')
axes[0].set_ylabel('Salario')
axes[0].set_title('Isolation Forest: Edad vs Salario', fontsize=12, color=C_DARK)
axes[0].legend()

# Scatter edad vs evaluacion
axes[1].scatter(normales['edad'], normales['evaluacion'], c=C_PRIMARY, alpha=0.4, s=20, label=f'Normal (n={len(normales)})')
axes[1].scatter(anomalos['edad'], anomalos['evaluacion'], c=C_DANGER, alpha=0.8, s=60, marker='X', label=f'Anomalia (n={len(anomalos)})')
axes[1].set_xlabel('Edad')
axes[1].set_ylabel('Evaluacion')
axes[1].set_title('Isolation Forest: Edad vs Evaluacion', fontsize=12, color=C_DARK)
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nMuestra de puntos anomalos:")
display(anomalos.head(10))

**Ejercicio 18:** Crear una columna con valores monetarios en formato string como "$1,234.56" y "$45,678.90". Limpiar estos strings eliminando el simbolo "$" y las comas, y convertir el resultado a tipo float. Mostrar la columna antes y despues de la conversion.

**Resolucion:**

1. Los valores monetarios como "$1,234.56" son strings que no se pueden usar directamente en calculos numericos.
2. **Paso 1:** Eliminar el simbolo "$" con `str.replace('$', '', regex=False)`.
3. **Paso 2:** Eliminar las comas con `str.replace(',', '', regex=False)`.
4. **Paso 3:** Convertir a float con `.astype(float)` o `pd.to_numeric()`.
5. Alternativa en un solo paso con regex: `str.replace(r'[\$,]', '', regex=True).astype(float)`.
6. Se verifica que el dtype cambie de `object` a `float64` y que los valores sean correctos.

In [ ]:
# --- Ejercicio 18: Limpiar columna "$1,234.56" y convertir a float ---

# Crear columna con formato monetario
np.random.seed(42)
precios_str = [f"${v:,.2f}" for v in np.random.lognormal(10, 0.8, 20)]
df_precios = pd.DataFrame({'precio_original': precios_str})

print("ANTES:")
print(f"Tipo: {df_precios['precio_original'].dtype}")
display(df_precios.head(10))

# Limpiar y convertir
df_precios['precio_limpio'] = (df_precios['precio_original']
                                .str.replace(r'[\$,]', '', regex=True)
                                .astype(float))

print("\nDESPUES:")
print(f"Tipo: {df_precios['precio_limpio'].dtype}")
display(df_precios.head(10))

# Visualizacion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(range(len(df_precios)), [0]*len(df_precios), color=C_PRIMARY)
for i, txt in enumerate(df_precios['precio_original']):
    axes[0].text(0.5, i, txt, va='center', fontsize=10, color=C_DARK)
axes[0].set_yticks(range(len(df_precios)))
axes[0].set_title('ANTES: strings monetarios', fontsize=12, color=C_DANGER)
axes[0].set_xlim(0, 1)

axes[1].barh(range(len(df_precios)), df_precios['precio_limpio'], color=C_SUCCESS, edgecolor='white')
axes[1].set_yticks(range(len(df_precios)))
axes[1].set_title('DESPUES: valores float', fontsize=12, color=C_SUCCESS)
axes[1].set_xlabel('Precio')

plt.tight_layout()
plt.show()

**Ejercicio 19:** Parsear la columna `fecha_ingreso` que contiene formatos mixtos: algunas fechas estan en formato "YYYY-MM-DD" y otras en "DD/MM/YYYY". Convertir todas a tipo datetime. Mostrar los valores antes y despues de la conversion, incluyendo los que no se pudieron parsear.

**Resolucion:**

1. `pd.to_datetime()` es la funcion principal para convertir strings a datetime.
2. Con formatos mixtos, se puede intentar el parseo automatico con `pd.to_datetime(col, infer_datetime_format=True, errors='coerce')`. El parametro `errors='coerce'` convierte los valores que no se pueden parsear en NaT (Not a Time).
3. Para mayor control, se aplica una funcion personalizada que intenta primero un formato y luego el otro:
   - Intentar `pd.to_datetime(x, format='%Y-%m-%d')` para "YYYY-MM-DD".
   - Si falla, intentar `pd.to_datetime(x, format='%d/%m/%Y')` para "DD/MM/YYYY".
4. Se verifica que el dtype resultante sea `datetime64[ns]`.
5. Se extraen componentes utiles: anno, mes, dia de la semana, con `.dt.year`, `.dt.month`, `.dt.dayofweek`.

In [ ]:
# --- Ejercicio 19: Parsear fecha_ingreso con formatos mixtos ---

def parsear_fecha(x):
    """Intenta parsear una fecha en formato YYYY-MM-DD o DD/MM/YYYY."""
    if pd.isna(x):
        return pd.NaT
    for fmt in ['%Y-%m-%d', '%d/%m/%Y']:
        try:
            return pd.to_datetime(x, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT

fechas_antes = df_dirty['fecha_ingreso'].copy()
fechas_despues = df_dirty['fecha_ingreso'].apply(parsear_fecha)

print(f"Tipo ANTES: {fechas_antes.dtype}")
print(f"Tipo DESPUES: {fechas_despues.dtype}")
print(f"NaT (no parseadas): {fechas_despues.isna().sum()}")

# Muestra comparativa
comparacion = pd.DataFrame({
    'original': fechas_antes,
    'parseada': fechas_despues
})
print("\nMuestra de conversion:")
display(comparacion.dropna(subset=['original']).head(15))

# Visualizacion: distribucion por mes
fig, ax = plt.subplots(figsize=(10, 5))
meses = fechas_despues.dt.month.dropna().astype(int)
meses.value_counts().sort_index().plot(kind='bar', ax=ax, color=C_PRIMARY, edgecolor='white')
ax.set_xlabel('Mes')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribucion de fecha_ingreso por mes (despues de parsear)', fontsize=12, color=C_DARK)
ax.set_xticklabels(['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic'], rotation=0)
plt.tight_layout()
plt.show()

**Ejercicio 20:** Validar los datos de `df_dirty` aplicando las siguientes reglas: edad debe estar en el rango [18, 70], salario debe ser mayor que 0, y evaluacion debe estar en el rango [1, 10]. Crear una columna `valido` que indique si la fila cumple todas las reglas. Mostrar el conteo de filas validas e invalidas.

**Resolucion:**

1. Se definen las reglas de validacion como expresiones booleanas:
   - `edad_ok`: `df['edad'].between(18, 70)` -- True si la edad esta entre 18 y 70 (inclusive). Los NaN devuelven False.
   - `salario_ok`: `df['salario'] > 0` -- True si el salario es positivo.
   - `eval_ok`: `df['evaluacion'].between(1, 10)` -- True si la evaluacion esta entre 1 y 10.
2. La columna `valido` es la conjuncion (AND) de todas las reglas: `edad_ok & salario_ok & eval_ok`.
3. Las filas con NaN en cualquiera de estas columnas se consideran invalidas automaticamente (NaN no cumple ninguna condicion).
4. Se muestran las filas invalidas para inspeccionar que regla violan.

In [ ]:
# --- Ejercicio 20: Validacion de rangos ---

df_val = df_dirty.copy()

# Reglas de validacion
df_val['edad_ok'] = df_val['edad'].between(18, 70)
df_val['salario_ok'] = df_val['salario'] > 0
df_val['eval_ok'] = df_val['evaluacion'].between(1, 10)
df_val['valido'] = df_val['edad_ok'] & df_val['salario_ok'] & df_val['eval_ok']

n_validos = df_val['valido'].sum()
n_invalidos = (~df_val['valido']).sum()

print(f"Filas validas: {n_validos} ({n_validos/len(df_val)*100:.1f}%)")
print(f"Filas invalidas: {n_invalidos} ({n_invalidos/len(df_val)*100:.1f}%)")

# Desglose por regla
print(f"\nDesglose de violaciones:")
print(f"  Edad fuera de [18, 70]: {(~df_val['edad_ok']).sum()}")
print(f"  Salario <= 0 o NaN:    {(~df_val['salario_ok']).sum()}")
print(f"  Evaluacion fuera [1,10]: {(~df_val['eval_ok']).sum()}")

print("\nMuestra de filas invalidas:")
display(df_val[~df_val['valido']][['id', 'edad', 'salario', 'evaluacion', 'edad_ok', 'salario_ok', 'eval_ok']].head(10))

# Visualizacion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
axes[0].pie([n_validos, n_invalidos], labels=['Validas', 'Invalidas'],
            colors=[C_SUCCESS, C_DANGER], autopct='%1.1f%%', startangle=90,
            textprops={'fontsize': 12})
axes[0].set_title('Proporcion de filas validas/invalidas', fontsize=12, color=C_DARK)

# Barplot por regla
reglas = ['edad_ok', 'salario_ok', 'eval_ok']
violaciones = [(~df_val[r]).sum() for r in reglas]
axes[1].bar(['Edad [18,70]', 'Salario > 0', 'Eval [1,10]'], violaciones,
            color=[C_DANGER, C_ORANGE, C_PRIMARY], edgecolor='white')
axes[1].set_ylabel('Filas que violan la regla')
axes[1].set_title('Violaciones por regla de validacion', fontsize=12, color=C_DARK)

plt.tight_layout()
plt.show()

del df_val

**Ejercicio 21:** Normalizar la columna `nombre` de `df_dirty` aplicando: (1) eliminar espacios al inicio y final (strip), (2) convertir a title case (primera letra de cada palabra en mayuscula), (3) reemplazar multiples espacios consecutivos por uno solo. Mostrar los valores unicos antes y despues.

**Resolucion:**

1. `str.strip()`: elimina espacios en blanco (y otros caracteres de espacio) al inicio y al final del string. Ejemplo: "  pedro martinez  " -> "pedro martinez".
2. `str.title()`: convierte la primera letra de cada palabra a mayuscula y el resto a minuscula. Ejemplo: "MARIA LOPEZ" -> "Maria Lopez".
3. `str.replace(r'\s+', ' ', regex=True)`: usa una expresion regular para reemplazar cualquier secuencia de uno o mas espacios (`\s+`) por un unico espacio.
4. El orden importa: primero strip (quitar extremos), luego replace (limpiar internos), luego title (normalizar case).
5. Se compara `nunique()` antes y despues para verificar que variantes como "MARIA LOPEZ", "  pedro martinez  " y "pedro Martinez" se unifican.

In [ ]:
# --- Ejercicio 21: Normalizar nombre ---

nombre_antes = df_dirty['nombre'].copy()
nombre_despues = (df_dirty['nombre']
                  .str.strip()
                  .str.replace(r'\s+', ' ', regex=True)
                  .str.title())

print("ANTES - Valores unicos:")
print(sorted(nombre_antes.dropna().unique()))
print(f"Total unicos: {nombre_antes.nunique()}")

print(f"\nDESPUES - Valores unicos:")
print(sorted(nombre_despues.dropna().unique()))
print(f"Total unicos: {nombre_despues.nunique()}")

# Visualizacion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

nombre_antes.value_counts(dropna=False).plot(kind='bar', ax=axes[0], color=C_DANGER, edgecolor='white')
axes[0].set_title(f'ANTES: {nombre_antes.nunique()} valores unicos', fontsize=12, color=C_DANGER)
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_ylabel('Frecuencia')

nombre_despues.value_counts(dropna=False).plot(kind='bar', ax=axes[1], color=C_SUCCESS, edgecolor='white')
axes[1].set_title(f'DESPUES: {nombre_despues.nunique()} valores unicos', fontsize=12, color=C_SUCCESS)
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

**Ejercicio 22:** Crear una columna de emails ficticios basados en los nombres de `df_dirty` (ej: "juan.garcia@empresa.com") y extraer el dominio de cada email utilizando expresiones regulares (regex).

**Resolucion:**

1. Se generan emails ficticios combinando el nombre normalizado con dominios aleatorios.
2. `str.extract(r'@(.+)')` usa una expresion regular para capturar todo lo que viene despues del "@":
   - `@`: coincide con el caracter "@" literal.
   - `(.+)`: grupo de captura que atrapa uno o mas caracteres despues del "@".
3. Alternativa: `str.split('@').str[1]` divide por "@" y toma la segunda parte.
4. Las expresiones regulares son una herramienta fundamental para extraer patrones de texto en limpieza de datos.

In [ ]:
# --- Ejercicio 22: Extraer dominio de emails con regex ---

np.random.seed(42)
dominios = ['empresa.com', 'corp.es', 'gmail.com', 'outlook.com', 'trabajo.org']

# Generar emails ficticios
nombre_norm = (df_dirty['nombre']
               .str.strip()
               .str.lower()
               .str.replace(r'\s+', '.', regex=True))

df_temp = pd.DataFrame({'nombre': df_dirty['nombre']})
df_temp['email'] = nombre_norm + '@' + np.random.choice(dominios, len(df_dirty))
df_temp.loc[df_dirty['nombre'].isna(), 'email'] = np.nan

# Extraer dominio con regex
df_temp['dominio'] = df_temp['email'].str.extract(r'@(.+)')

print("Muestra de emails y dominios extraidos:")
display(df_temp.dropna().head(10))

# Visualizacion
fig, ax = plt.subplots(figsize=(10, 5))
df_temp['dominio'].value_counts().plot(kind='bar', ax=ax, color=C_PRIMARY, edgecolor='white')
ax.set_title('Distribucion de dominios extraidos', fontsize=12, color=C_DARK)
ax.set_xlabel('Dominio')
ax.set_ylabel('Frecuencia')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

del df_temp

**Ejercicio 23:** Estandarizar la columna `departamento` de `df_dirty`: convertir todo a title case y corregir las inconsistencias (Ventas, ventas, VENTAS deben ser todas "Ventas"). Mostrar los valores unicos antes y despues, y la distribucion resultante.

**Resolucion:**

1. `str.strip()`: elimina espacios al inicio y final.
2. `str.title()`: convierte a Title Case. Ejemplo: "ventas" -> "Ventas", "VENTAS" -> "Ventas", "it" -> "It".
3. Para casos especiales como "IT" (que title case convertiria a "It"), se puede usar un diccionario de mapeo: `df['col'].replace({'It': 'IT', 'Rrhh': 'RRHH'})`.
4. Se verifica con `value_counts()` que todas las variantes se hayan unificado correctamente.
5. La estandarizacion de categorias es esencial antes de aplicar encoding o agrupaciones.

In [ ]:
# --- Ejercicio 23: Estandarizar departamento ---

depto_antes = df_dirty['departamento'].copy()
depto_despues = (df_dirty['departamento']
                 .str.strip()
                 .str.title()
                 .replace({'It': 'IT', 'Rrhh': 'RRHH'}))

print("ANTES - Valores unicos:")
print(sorted(depto_antes.dropna().unique()))
print(f"Total unicos: {depto_antes.nunique()}")

print(f"\nDESPUES - Valores unicos:")
print(sorted(depto_despues.dropna().unique()))
print(f"Total unicos: {depto_despues.nunique()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

depto_antes.value_counts(dropna=False).plot(kind='bar', ax=axes[0], color=C_DANGER, edgecolor='white')
axes[0].set_title(f'ANTES: {depto_antes.nunique()} categorias', fontsize=12, color=C_DANGER)
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_ylabel('Frecuencia')

depto_despues.value_counts(dropna=False).plot(kind='bar', ax=axes[1], color=C_SUCCESS, edgecolor='white')
axes[1].set_title(f'DESPUES: {depto_despues.nunique()} categorias', fontsize=12, color=C_SUCCESS)
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

**Ejercicio 24:** Aplicar One-Hot Encoding a la columna `ciudad` (previamente normalizada a title case). Usar `pd.get_dummies()` con `drop_first=True` para evitar la trampa de variables dummy. Mostrar las primeras filas del resultado y explicar por que se elimina una columna.

**Resolucion:**

1. **One-Hot Encoding:** Crea una columna binaria (0/1) por cada categoria unica. Si `ciudad` tiene 4 valores unicos (Madrid, Barcelona, Sevilla, Valencia), se crean 4 columnas.
2. **Trampa de variables dummy (dummy trap):** Si se incluyen todas las columnas dummy, se genera multicolinealidad perfecta (la suma de todas las dummies siempre es 1). Esto causa problemas en modelos lineales.
3. `drop_first=True`: elimina la primera categoria (que se convierte en la "referencia"). Si todas las dummies son 0, la observacion pertenece a la categoria eliminada.
4. `pd.get_dummies(df, columns=['ciudad'], drop_first=True)` aplica el encoding directamente sobre el DataFrame.
5. Alternativa con sklearn: `OneHotEncoder(drop='first', sparse_output=False)`.

In [ ]:
# --- Ejercicio 24: One-Hot Encoding de ciudad ---

ciudad_norm = df_dirty['ciudad'].str.strip().str.title()

print("ANTES - Valores unicos de ciudad:")
print(ciudad_norm.value_counts().to_string())

# Sin drop_first
dummies_full = pd.get_dummies(ciudad_norm, prefix='ciudad')
print(f"\nOne-Hot SIN drop_first ({dummies_full.shape[1]} columnas):")
display(dummies_full.head(8))

# Con drop_first
dummies_drop = pd.get_dummies(ciudad_norm, prefix='ciudad', drop_first=True)
print(f"\nOne-Hot CON drop_first ({dummies_drop.shape[1]} columnas):")
display(dummies_drop.head(8))

# Visualizacion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Antes: distribucion de ciudad
ciudad_norm.value_counts().plot(kind='bar', ax=axes[0], color=C_PRIMARY, edgecolor='white')
axes[0].set_title('ANTES: distribucion de ciudad', fontsize=12, color=C_DARK)
axes[0].tick_params(axis='x', rotation=45)

# Despues: heatmap de dummies
sns.heatmap(dummies_drop.head(20).T, cmap='Blues', cbar=False, ax=axes[1],
            linewidths=0.5, annot=True, fmt='d')
axes[1].set_title('DESPUES: One-Hot (primeras 20 filas)', fontsize=12, color=C_DARK)
axes[1].set_xlabel('Indice de fila')

plt.tight_layout()
plt.show()

**Ejercicio 25:** Aplicar Ordinal Encoding a la columna `satisfaccion` con el orden logico: bajo=1, medio=2, alto=3. Primero normalizar los valores (strip, lower) para unificar variantes. Mostrar el mapeo y la distribucion resultante.

**Resolucion:**

1. **Ordinal Encoding:** Asigna un entero a cada categoria respetando un orden logico. A diferencia de Label Encoding (orden arbitrario), aqui el orden tiene significado: bajo < medio < alto.
2. Primero se normaliza: `str.strip().str.lower()` para unificar "bajo", "Bajo", "ALTO" en "bajo", "alto".
3. Se define el mapeo manualmente: `{'bajo': 1, 'medio': 2, 'alto': 3}`.
4. Se aplica con `df['col'].map(mapeo)`.
5. Alternativa con sklearn: `OrdinalEncoder(categories=[['bajo', 'medio', 'alto']])`.
6. Los valores NaN se mantienen como NaN tras el mapeo (no aparecen en el diccionario).

In [ ]:
# --- Ejercicio 25: Ordinal Encoding de satisfaccion ---

satis_norm = df_dirty['satisfaccion'].str.strip().str.lower()

print("ANTES - Valores unicos:")
print(satis_norm.value_counts(dropna=False).to_string())

# Mapeo ordinal
mapeo_ordinal = {'bajo': 1, 'medio': 2, 'alto': 3}
satis_encoded = satis_norm.map(mapeo_ordinal)

print(f"\nMapeo aplicado: {mapeo_ordinal}")
print(f"\nDESPUES - Valores unicos:")
print(satis_encoded.value_counts(dropna=False).to_string())

# Muestra
muestra = pd.DataFrame({
    'original': df_dirty['satisfaccion'],
    'normalizado': satis_norm,
    'encoded': satis_encoded
})
display(muestra.head(10))

# Visualizacion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

satis_norm.value_counts(dropna=False).plot(kind='bar', ax=axes[0], color=C_PRIMARY, edgecolor='white')
axes[0].set_title('ANTES: satisfaccion (categorica)', fontsize=12, color=C_DARK)
axes[0].tick_params(axis='x', rotation=0)

satis_encoded.value_counts(dropna=False).sort_index().plot(kind='bar', ax=axes[1],
    color=[C_DANGER, C_ORANGE, C_SUCCESS, '#cccccc'], edgecolor='white')
axes[1].set_title('DESPUES: satisfaccion (ordinal)', fontsize=12, color=C_SUCCESS)
axes[1].set_xticklabels(['1 (bajo)', '2 (medio)', '3 (alto)', 'NaN'], rotation=0)

plt.tight_layout()
plt.show()

**Ejercicio 26:** Aplicar Target Encoding a la columna `departamento` usando la media de `evaluacion` por departamento. Es decir, reemplazar cada valor de departamento por la media de evaluacion de ese departamento. Normalizar departamento antes de aplicar el encoding.

**Resolucion:**

1. **Target Encoding:** Reemplaza cada categoria por la media de la variable objetivo (target) para esa categoria. Es una forma de encoding que captura la relacion entre la variable categorica y el target.
2. Se calcula la media de `evaluacion` agrupada por `departamento` normalizado: `df.groupby('depto_norm')['evaluacion'].mean()`.
3. Se crea un diccionario de mapeo con esas medias y se aplica con `df['depto_norm'].map(mapeo)`.
4. **Riesgo de data leakage:** En produccion, el target encoding debe calcularse solo sobre el conjunto de entrenamiento y aplicarse al de test. Calcularlo sobre todo el dataset introduce fuga de informacion.
5. Para mitigar el overfitting, se pueden usar tecnicas como leave-one-out encoding o agregar ruido (smoothing).

In [ ]:
# --- Ejercicio 26: Target Encoding de departamento ---

df_temp = df_dirty[['departamento', 'evaluacion']].copy()
df_temp['depto_norm'] = df_temp['departamento'].str.strip().str.title().replace({'It': 'IT', 'Rrhh': 'RRHH'})

# Calcular media de evaluacion por departamento
target_map = df_temp.groupby('depto_norm')['evaluacion'].mean()
print("Media de evaluacion por departamento:")
print(target_map.round(3).to_string())

# Aplicar target encoding
df_temp['depto_target_enc'] = df_temp['depto_norm'].map(target_map)

print(f"\nMuestra del resultado:")
display(df_temp.head(10))

# Visualizacion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

target_map.sort_values().plot(kind='barh', ax=axes[0], color=C_PRIMARY, edgecolor='white')
axes[0].set_xlabel('Media de evaluacion')
axes[0].set_title('Target Encoding: media de evaluacion por departamento', fontsize=11, color=C_DARK)

axes[1].hist(df_temp['depto_target_enc'].dropna(), bins=20, color=C_SUCCESS, alpha=0.7, edgecolor='white')
axes[1].set_xlabel('Valor encoded')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribucion de departamento tras Target Encoding', fontsize=11, color=C_SUCCESS)

plt.tight_layout()
plt.show()

del df_temp

**Ejercicio 27:** Comparar tres metodos de escalado en la columna `salario` (con outliers): MinMaxScaler, StandardScaler y RobustScaler. Mostrar tres histogramas lado a lado con las distribuciones resultantes y discutir cual es mas apropiado en presencia de outliers.

**Resolucion:**

1. **MinMaxScaler:** `x_scaled = (x - x_min) / (x_max - x_min)`. Escala al rango [0, 1]. Un outlier extremo comprime todos los demas valores cerca de 0.
2. **StandardScaler:** `x_scaled = (x - media) / std`. Centra en 0 con desviacion 1. Los outliers afectan la media y la desviacion, distorsionando el escalado.
3. **RobustScaler:** `x_scaled = (x - mediana) / IQR`. Usa mediana e IQR en lugar de media y std. Es robusto a outliers porque estas estadisticas no se ven afectadas por valores extremos.
4. Se aplica cada scaler con `.fit_transform(X)` donde X es un array 2D.
5. **Conclusion:** Con outliers, RobustScaler es la opcion mas segura. MinMaxScaler es el mas afectado.

In [ ]:
# --- Ejercicio 27: Comparar MinMax vs Standard vs Robust en salario ---

salario = df_dirty['salario'].dropna().values.reshape(-1, 1)

scaler_mm = MinMaxScaler()
scaler_ss = StandardScaler()
scaler_rb = RobustScaler()

sal_minmax = scaler_mm.fit_transform(salario).flatten()
sal_standard = scaler_ss.fit_transform(salario).flatten()
sal_robust = scaler_rb.fit_transform(salario).flatten()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(sal_minmax, bins=40, color=C_PRIMARY, alpha=0.7, edgecolor='white')
axes[0].set_title('MinMaxScaler [0, 1]', fontsize=12, color=C_PRIMARY)
axes[0].set_xlabel('Valor escalado')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(0, color=C_DARK, linestyle=':', alpha=0.5)
axes[0].axvline(1, color=C_DARK, linestyle=':', alpha=0.5)

axes[1].hist(sal_standard, bins=40, color=C_ORANGE, alpha=0.7, edgecolor='white')
axes[1].set_title('StandardScaler (media=0, std=1)', fontsize=12, color=C_ORANGE)
axes[1].set_xlabel('Valor escalado')
axes[1].axvline(0, color=C_DARK, linestyle='--', alpha=0.5, label='media=0')
axes[1].legend()

axes[2].hist(sal_robust, bins=40, color=C_SUCCESS, alpha=0.7, edgecolor='white')
axes[2].set_title('RobustScaler (mediana, IQR)', fontsize=12, color=C_SUCCESS)
axes[2].set_xlabel('Valor escalado')
axes[2].axvline(0, color=C_DARK, linestyle='--', alpha=0.5, label='mediana=0')
axes[2].legend()

plt.suptitle('Comparacion de escaladores en "salario" (con outliers)', fontsize=14, color=C_DARK, y=1.02)
plt.tight_layout()
plt.show()

# Estadisticas
print("\n--- Estadisticas de cada escalado ---")
for nombre, datos in [('MinMax', sal_minmax), ('Standard', sal_standard), ('Robust', sal_robust)]:
    print(f"  {nombre:10s}: min={datos.min():.3f}, max={datos.max():.3f}, media={datos.mean():.3f}, std={datos.std():.3f}")

**Ejercicio 28:** Usar `ColumnTransformer` de scikit-learn para aplicar transformaciones diferentes a columnas numericas y categoricas simultaneamente: escalar las numericas (`edad`, `salario`, `evaluacion`) con StandardScaler y aplicar One-Hot Encoding a las categoricas (`departamento`, `ciudad`). Primero preparar los datos eliminando NaN.

**Resolucion:**

1. **ColumnTransformer:** Permite aplicar diferentes transformadores a diferentes subconjuntos de columnas en un solo paso.
2. Se define como una lista de tuplas `(nombre, transformador, columnas)`:
   - `('num', StandardScaler(), cols_numericas)`: escala las columnas numericas.
   - `('cat', OneHotEncoder(drop='first', sparse_output=False), cols_categoricas)`: aplica one-hot a las categoricas.
3. `ct.fit_transform(X)` aplica todas las transformaciones y devuelve un array con las columnas transformadas concatenadas.
4. `ct.get_feature_names_out()` devuelve los nombres de las columnas resultantes.
5. Se preparan los datos eliminando filas con NaN en las columnas relevantes (o imputando previamente).
6. El resultado es una matriz lista para alimentar un modelo de ML.

In [ ]:
# --- Ejercicio 28: ColumnTransformer ---

# Preparar datos: normalizar categoricas y eliminar NaN
df_ct = df_dirty.copy()
df_ct['departamento'] = df_ct['departamento'].str.strip().str.title().replace({'It': 'IT', 'Rrhh': 'RRHH'})
df_ct['ciudad'] = df_ct['ciudad'].str.strip().str.title()

cols_num = ['edad', 'salario', 'evaluacion']
cols_cat = ['departamento', 'ciudad']
df_ct = df_ct[cols_num + cols_cat].dropna()

print(f"Shape tras eliminar NaN: {df_ct.shape}")
print(f"Columnas numericas: {cols_num}")
print(f"Columnas categoricas: {cols_cat}")

# Definir ColumnTransformer
ct = ColumnTransformer([
    ('num', StandardScaler(), cols_num),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), cols_cat)
])

# Aplicar
X_transformed = ct.fit_transform(df_ct)
feature_names = ct.get_feature_names_out()

df_result = pd.DataFrame(X_transformed, columns=feature_names)
print(f"\nShape resultante: {df_result.shape}")
print(f"Columnas: {list(feature_names)}")

display(df_result.head(10))

# Visualizacion
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Numericas escaladas
for i, col in enumerate(cols_num):
    axes[0].hist(df_result[f'num__{col}'], bins=30, alpha=0.5,
                 label=col, color=[C_PRIMARY, C_DANGER, C_SUCCESS][i])
axes[0].set_title('Columnas numericas escaladas (StandardScaler)', fontsize=11, color=C_DARK)
axes[0].legend()
axes[0].set_xlabel('Valor escalado')

# Categoricas one-hot
cat_cols = [c for c in feature_names if c.startswith('cat__')]
cat_sums = df_result[cat_cols].sum()
cat_sums.plot(kind='bar', ax=axes[1], color=C_ORANGE, edgecolor='white')
axes[1].set_title('Columnas categoricas (One-Hot sumas)', fontsize=11, color=C_DARK)
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylabel('Suma de 1s')

plt.tight_layout()
plt.show()

del df_ct

**Ejercicio 29:** Construir un pipeline completo con scikit-learn que aplique: para columnas numericas, `SimpleImputer(strategy='median')` seguido de `StandardScaler`; para columnas categoricas, `SimpleImputer(strategy='most_frequent')` seguido de `OneHotEncoder`. Combinar ambos con `ColumnTransformer` y aplicar a `df_dirty`.

**Resolucion:**

1. **Pipeline:** Encadena transformadores en secuencia. Cada paso recibe la salida del anterior.
   - `Pipeline([('imputer', SimpleImputer(...)), ('scaler', StandardScaler())])`.
2. **Pipeline numerico:** `SimpleImputer(strategy='median')` rellena NaN con la mediana, luego `StandardScaler()` escala a media=0, std=1.
3. **Pipeline categorico:** `SimpleImputer(strategy='most_frequent')` rellena NaN con la moda, luego `OneHotEncoder(handle_unknown='ignore', sparse_output=False)` codifica las categorias.
4. **ColumnTransformer:** Combina ambos pipelines, aplicando cada uno a su subconjunto de columnas.
5. El resultado es un array numpy limpio, sin NaN, con todas las columnas transformadas y listo para un modelo.
6. Ventaja: todo el preprocesamiento queda encapsulado en un solo objeto reproducible.

In [ ]:
# --- Ejercicio 29: Pipeline completo con ColumnTransformer ---

# Normalizar categoricas primero
df_pipe = df_dirty.copy()
df_pipe['departamento'] = df_pipe['departamento'].str.strip().str.title().replace({'It': 'IT', 'Rrhh': 'RRHH'})
df_pipe['ciudad'] = df_pipe['ciudad'].str.strip().str.title()
df_pipe['satisfaccion'] = df_pipe['satisfaccion'].str.strip().str.lower()

cols_num = ['edad', 'salario', 'evaluacion']
cols_cat = ['departamento', 'ciudad', 'satisfaccion']

# Pipeline numerico
pipe_num = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline categorico
pipe_cat = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first'))
])

# ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', pipe_num, cols_num),
    ('cat', pipe_cat, cols_cat)
])

# Aplicar
X = df_pipe[cols_num + cols_cat]
X_transformed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

df_result = pd.DataFrame(X_transformed, columns=feature_names)

print(f"Shape original: {X.shape}")
print(f"Shape transformado: {df_result.shape}")
print(f"NaN en resultado: {df_result.isnull().sum().sum()}")
print(f"\nColumnas resultantes ({len(feature_names)}):")
print(list(feature_names))

display(df_result.head(10))

# Visualizacion
fig, ax = plt.subplots(figsize=(16, 5))
ax.barh(range(len(feature_names)), df_result.mean().values, color=C_PRIMARY, edgecolor='white')
ax.set_yticks(range(len(feature_names)))
ax.set_yticklabels(feature_names, fontsize=8)
ax.set_xlabel('Valor medio')
ax.set_title('Media de cada feature tras el pipeline', fontsize=13, color=C_DARK)
plt.tight_layout()
plt.show()

del df_pipe

**Ejercicio 30:** Limpieza end-to-end. Tomar `df_dirty` original y producir `df_clean` listo para modelo, aplicando TODOS los pasos anteriores en secuencia: (1) eliminar duplicados, (2) normalizar strings, (3) parsear fechas, (4) validar rangos y marcar invalidos, (5) tratar outliers con IQR, (6) imputar nulos, (7) encoding categorico, (8) escalar numericas. Mostrar un resumen comparativo del dataset antes y despues.

**Resolucion:**

El pipeline end-to-end aplica cada tecnica aprendida en orden logico:

1. **Duplicados:** `drop_duplicates()` elimina filas identicas. Se hace primero para no procesar datos redundantes.
2. **Strings:** Normalizar `nombre`, `departamento`, `ciudad`, `satisfaccion` con strip + title/lower + replace de inconsistencias.
3. **Fechas:** Parsear `fecha_ingreso` con la funcion personalizada que maneja formatos mixtos. Extraer anno, mes, dia de la semana.
4. **Validacion:** Aplicar reglas de negocio (edad [18,70], salario > 0, evaluacion [1,10]). Reemplazar invalidos por NaN.
5. **Outliers:** Aplicar IQR en `salario` para winsorizar valores extremos.
6. **Imputacion:** Mediana para numericas, moda para categoricas.
7. **Encoding:** Ordinal para `satisfaccion`, One-Hot para `departamento` y `ciudad`.
8. **Escalado:** StandardScaler para todas las columnas numericas.

El resultado es un DataFrame sin NaN, sin duplicados, con tipos correctos, valores en rango y listo para alimentar un modelo.

In [ ]:
# --- Ejercicio 30: Limpieza end-to-end ---

df = df_dirty.copy()
print(f"INICIO: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"  NaN totales: {df.isnull().sum().sum()}")

# === PASO 1: Eliminar duplicados ===
n_antes = len(df)
df = df.drop_duplicates()
print(f"\n[1] Duplicados eliminados: {n_antes - len(df)} -> quedan {len(df)} filas")

# === PASO 2: Normalizar strings ===
df['nombre'] = df['nombre'].str.strip().str.replace(r'\s+', ' ', regex=True).str.title()
df['departamento'] = df['departamento'].str.strip().str.title().replace({'It': 'IT', 'Rrhh': 'RRHH'})
df['ciudad'] = df['ciudad'].str.strip().str.title()
df['satisfaccion'] = df['satisfaccion'].str.strip().str.lower()
print(f"[2] Strings normalizados")

# === PASO 3: Parsear fechas ===
def parsear_fecha(x):
    if pd.isna(x):
        return pd.NaT
    for fmt in ['%Y-%m-%d', '%d/%m/%Y']:
        try:
            return pd.to_datetime(x, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT

df['fecha_ingreso'] = df['fecha_ingreso'].apply(parsear_fecha)
df['mes_ingreso'] = df['fecha_ingreso'].dt.month
df['dia_semana'] = df['fecha_ingreso'].dt.dayofweek
print(f"[3] Fechas parseadas, extraidos mes y dia de semana")

# === PASO 4: Validar rangos ===
df.loc[~df['edad'].between(18, 70), 'edad'] = np.nan
df.loc[df['salario'] <= 0, 'salario'] = np.nan
df.loc[~df['evaluacion'].between(1, 10), 'evaluacion'] = np.nan
print(f"[4] Valores fuera de rango reemplazados por NaN")

# === PASO 5: Tratar outliers (winsorizar salario con IQR) ===
Q1 = df['salario'].quantile(0.25)
Q3 = df['salario'].quantile(0.75)
IQR = Q3 - Q1
lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR
n_out = ((df['salario'] < lim_inf) | (df['salario'] > lim_sup)).sum()
df['salario'] = df['salario'].clip(lower=lim_inf, upper=lim_sup)
print(f"[5] Outliers en salario winsorizados: {n_out} valores recortados")

# === PASO 6: Imputar nulos ===
df['edad'] = df['edad'].fillna(df['edad'].median())
df['salario'] = df['salario'].fillna(df['salario'].median())
df['evaluacion'] = df['evaluacion'].fillna(df['evaluacion'].median())
df['nombre'] = df['nombre'].fillna('Desconocido')
df['departamento'] = df['departamento'].fillna(df['departamento'].mode()[0])
df['ciudad'] = df['ciudad'].fillna(df['ciudad'].mode()[0])
df['satisfaccion'] = df['satisfaccion'].fillna(df['satisfaccion'].mode()[0])
df['mes_ingreso'] = df['mes_ingreso'].fillna(df['mes_ingreso'].median())
df['dia_semana'] = df['dia_semana'].fillna(df['dia_semana'].median())
print(f"[6] Nulos imputados. NaN restantes: {df.isnull().sum().sum()}")

# === PASO 7: Encoding categorico ===
mapeo_satis = {'bajo': 1, 'medio': 2, 'alto': 3}
df['satisfaccion_ord'] = df['satisfaccion'].map(mapeo_satis)
df = pd.get_dummies(df, columns=['departamento', 'ciudad'], drop_first=True)
print(f"[7] Encoding aplicado: ordinal (satisfaccion), one-hot (departamento, ciudad)")

# === PASO 8: Escalar numericas ===
cols_to_scale = ['edad', 'salario', 'evaluacion', 'satisfaccion_ord', 'mes_ingreso', 'dia_semana']
scaler = StandardScaler()
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])
print(f"[8] Columnas numericas escaladas con StandardScaler")

# Eliminar columnas auxiliares
df_clean = df.drop(columns=['id', 'nombre', 'satisfaccion', 'fecha_ingreso'], errors='ignore')

print(f"\n{'='*60}")
print(f"RESULTADO FINAL: {df_clean.shape[0]} filas x {df_clean.shape[1]} columnas")
print(f"NaN totales: {df_clean.isnull().sum().sum()}")
print(f"Tipos: {df_clean.dtypes.value_counts().to_dict()}")
print(f"Columnas: {list(df_clean.columns)}")

display(df_clean.head(10))

# === VISUALIZACION COMPARATIVA ===
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Comparar shape
labels_shape = ['Filas (dirty)', 'Filas (clean)', 'Cols (dirty)', 'Cols (clean)']
vals_shape = [df_dirty.shape[0], df_clean.shape[0], df_dirty.shape[1], df_clean.shape[1]]
colors_shape = [C_DANGER, C_SUCCESS, C_DANGER, C_SUCCESS]
bars = axes[0, 0].bar(labels_shape, vals_shape, color=colors_shape, edgecolor='white')
axes[0, 0].bar_label(bars, fontsize=10)
axes[0, 0].set_title('Dimensiones: antes vs despues', fontsize=12, color=C_DARK)

# NaN comparison
nulls_dirty = df_dirty.select_dtypes(include=[np.number]).isnull().sum()
axes[0, 1].bar(nulls_dirty.index, nulls_dirty.values, color=C_DANGER, alpha=0.7, label='Dirty')
axes[0, 1].bar(nulls_dirty.index, [0]*len(nulls_dirty), color=C_SUCCESS, alpha=0.7, label='Clean (0 NaN)')
axes[0, 1].set_title('Valores nulos por columna numerica', fontsize=12, color=C_DARK)
axes[0, 1].legend()
axes[0, 1].tick_params(axis='x', rotation=45)

# Distribucion de salario antes vs despues
axes[1, 0].hist(df_dirty['salario'].dropna(), bins=30, color=C_DANGER, alpha=0.5, label='Dirty', edgecolor='white')
axes[1, 0].set_title('Salario: distribucion original (con outliers)', fontsize=12, color=C_DANGER)
axes[1, 0].legend()
axes[1, 0].set_xlabel('Salario')

salario_idx = cols_to_scale.index('salario')
axes[1, 1].hist(df_clean['salario'], bins=30, color=C_SUCCESS, alpha=0.7, edgecolor='white', label='Clean (escalado)')
axes[1, 1].set_title('Salario: distribucion limpia (escalada)', fontsize=12, color=C_SUCCESS)
axes[1, 1].legend()
axes[1, 1].set_xlabel('Salario (escalado)')

plt.suptitle('Resumen: df_dirty vs df_clean', fontsize=15, color=C_DARK, y=1.02)
plt.tight_layout()
plt.show()